# 07 - Classifier sweep

Mirrors `notebooks_withseed/06_algorithm_sweep`. Answers whether logistic regression is
underselling the frozen parcel features, or whether some other classifier does better.

PCA front-end by default: it makes the heavy classifiers fast and, on the sensor-level data, barely
changed the answer.

In [5]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')
import numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
OSF, OUT = '../data/preprocessed', '../results_parcel'
MODEL, SUFFIX = 'tiny', 'cortical'
cv = StratifiedKFold(5, shuffle=True, random_state=123)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, AdaBoostClassifier,
                              BaggingClassifier, GradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import RidgeClassifier, SGDClassifier, PassiveAggressiveClassifier

labels = np.load(f'{OUT}/parcel_labels_{SUFFIX}.npy')
X = np.load(f'{OUT}/parcel_frozen_{MODEL}_{SUFFIX}.npy')
PCA_N, S = 50, 123
eff = PCA_N if X.shape[1] > PCA_N else None
print(f'{X.shape} | PCA front-end: {eff}')

def build(clf, scale):
    steps = []
    if eff: steps.append(PCA(n_components=eff, random_state=S))
    if scale: steps.append(StandardScaler())
    steps.append(clf)
    return make_pipeline(*steps) if len(steps) > 1 else clf

(161, 4096) | PCA front-end: 50


In [6]:
SPEC = [
 ('LR (L2)',        LogisticRegression(max_iter=2000), False),
 ('LR L1',          LogisticRegression(penalty='l1', solver='saga', max_iter=4000), True),
 ('LR ElasticNet',  LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=4000), True),
 ('Ridge',          RidgeClassifier(), True),
 ('SGD (log)',      SGDClassifier(loss='log_loss', random_state=S), True),
 ('PassiveAggr',    PassiveAggressiveClassifier(random_state=S), True),
 ('Linear SVM',     LinearSVC(max_iter=5000, random_state=S), True),
 ('RBF SVM',        SVC(kernel='rbf', random_state=S), True),
 ('Poly SVM (d3)',  SVC(kernel='poly', degree=3, random_state=S), True),
 ('Sigmoid SVM',    SVC(kernel='sigmoid', random_state=S), True),
 ('kNN k=5',        KNeighborsClassifier(n_neighbors=5), True),
 ('kNN k=15',       KNeighborsClassifier(n_neighbors=15), True),
 ('NearestCentroid',NearestCentroid(), True),
 ('GaussianNB',     GaussianNB(), True),
 ('LDA (shrink)',   LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'), True),
 ('RandomForest',   RandomForestClassifier(n_estimators=300, random_state=S, n_jobs=-1), False),
 ('ExtraTrees',     ExtraTreesClassifier(n_estimators=300, random_state=S, n_jobs=-1), False),
 ('GradBoost',      GradientBoostingClassifier(random_state=S), False),
 ('HistGradBoost',  HistGradientBoostingClassifier(random_state=S), False),
 ('AdaBoost',       AdaBoostClassifier(random_state=S), False),
 ('Bagging',        BaggingClassifier(n_estimators=50, random_state=S, n_jobs=-1), False),
 ('MLP (64)',       MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=S), True),
 ('MLP (128,64)',   MLPClassifier(hidden_layer_sizes=(128,64), max_iter=1000, random_state=S), True),
]
rows = []
for name, clf, scale in SPEC:
    r = cross_validate(build(clf, scale), X, labels, cv=cv, scoring=['accuracy','roc_auc'])
    a, u = r['test_accuracy'].mean(), r['test_roc_auc'].mean()
    rows.append((name, a, u)); print(f'  {name:20s} acc {a:.3f}  AUC {u:.3f}', flush=True)

rows.sort(key=lambda r: -r[1])
print(f'\n=== ranked (PCA={eff}) ===')
for name, a, u in rows:
    star = '  <- LR reference' if name == 'LR (L2)' else ''
    print(f'  {name:20s} acc {a:.3f}  AUC {u:.3f}{star}')

  LR (L2)              acc 0.857  AUC 0.922
  LR L1                acc 0.825  AUC 0.893
  LR ElasticNet        acc 0.825  AUC 0.890
  Ridge                acc 0.851  AUC 0.901
  SGD (log)            acc 0.770  AUC 0.864
  PassiveAggr          acc 0.789  AUC 0.859
  Linear SVM           acc 0.782  AUC 0.854
  RBF SVM              acc 0.826  AUC 0.902
  Poly SVM (d3)        acc 0.528  AUC 0.861
  Sigmoid SVM          acc 0.832  AUC 0.902
  kNN k=5              acc 0.708  AUC 0.735
  kNN k=15             acc 0.745  AUC 0.818
  NearestCentroid      acc 0.857  AUC 0.915
  GaussianNB           acc 0.813  AUC 0.864
  LDA (shrink)         acc 0.844  AUC 0.915
  RandomForest         acc 0.838  AUC 0.901
  ExtraTrees           acc 0.875  AUC 0.898
  GradBoost            acc 0.826  AUC 0.907
  HistGradBoost        acc 0.844  AUC 0.895
  AdaBoost             acc 0.826  AUC 0.894
  Bagging              acc 0.832  AUC 0.895
  MLP (64)             acc 0.813  AUC 0.860
  MLP (128,64)         acc 0.826